# XGLUE - Temporary Small Graph Inspection
Quick visual sanity checks for prepared XGLUE data, Joern graph shards, and spectral features.


In [ ]:
from pathlib import Path
import sys


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "spectral_code").exists() and (candidate / "pipelines").exists():
            return candidate
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from IPython.display import display

from spectral_code.evaluation.notebook_helpers import (
    artifact_status_dataframe,
    configure_notebook_style,
    dataset_overview_rows,
    display_clone_nonclone_pair_inspection,
    display_code_graph_side_by_side_examples,
    display_dataset_overview,
    display_global_threshold_tuning_summary,
    display_pipeline_validation,
    display_similarity_distribution_report,
    display_statistical_distribution_plots,
    display_tuning_report,
    graph_manifest_summary_dataframe,
    load_pairs_for_spec,
    load_tuning_results,
    notebook_output_dir,
    pair_stats_dataframe,
    plot_graph_coverage_from_timing,
    plot_graph_manifest_summary,
    plot_timing_stats,
    timing_stats_dataframe,
    xglue_spec,
)

configure_notebook_style()
spec = xglue_spec()
GRAPH_TYPES = ["ast", "cfg", "ddg", "pdg", "cpg"]
ARTIFACT_DIR = notebook_output_dir(spec)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {spec.data_dir}")
print(f"Output root: {spec.output_root}")
print(f"Notebook artifacts: {ARTIFACT_DIR}")


## Artifact Readiness


In [ ]:
display(artifact_status_dataframe(spec))
summary = graph_manifest_summary_dataframe(spec)
if not summary.empty:
    display(summary)
    plot_graph_manifest_summary(spec)
else:
    print("Graph and spectral summaries are not available yet.")


## Random Code vs. Graph Examples


In [ ]:
graph_manifest = spec.output_root / "clean_graphs" / "graph_shards_manifest.json"
if graph_manifest.exists():
    selected_ids = display_code_graph_side_by_side_examples(
        spec,
        n_examples=3,
        graph_type="cpg",
        seed=42,
        max_code_lines=38,
        max_nodes=80,
    )
    selected_ids
else:
    print(f"Run 02_extract_graphs.py first. Missing: {graph_manifest}")


## Clone and Non-clone Pair Inspection


In [ ]:
if spec.features_manifest.exists():
    inspected_pairs = display_clone_nonclone_pair_inspection(
        spec,
        graph_type="cpg",
        per_label=2,
        seed=42,
        max_code_lines=70,
        max_nodes=80,
    )
    inspected_pairs
else:
    print(f"Run 03_extract_spectral_features.py first. Missing: {spec.features_manifest}")
